In [92]:
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'
import sys
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')

In [93]:
import os,cv2
os.chdir(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')

from Split import VertPart
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

In [94]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [95]:
from OCR import Clova
api_url='https://1srlcnzf08.apigw.ntruss.com/custom/v1/2412/9a859f9b3a7d952aad17f388d1a445041f8aef0f1eccc6fcce8d3a856272fcbd/general'
secret_key='eEhyV0NGRlRLVXpZVWZnWFNDamRVaWFYZ1NSQ1NKSFI='


In [96]:
import sys, json, os, cv2
import pandas as pd

Year,Showa='1941','16'
Quality='HQ' #HQ or Line#

df = pd.read_csv(origin+'Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)



In [97]:
StepAError,StepBError,MainError=[],[],[]
Dict={}
Data=[]
for Page in range(15, 125, 10):
    #Step A
#    try:        
    file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
    img=cv2.imread(os.path.join(origin,file_path))

    #Convert to book page
    BookPage=2*Page-14
    Rows=df[(df['Page']==BookPage)]
    NxRows=df[(df['Page']==BookPage+1)]
    if len(Rows)==0:
        print('No Office at Right Side. Page'+str(BookPage))
    if len(NxRows)==0:
        print('No Office at Left Side. Page'+str(BookPage+1))

    texts=Vision(img,'zh',client)
    textsCLOVA=Clova(origin,Page,api_url,secret_key,Year,Quality)
#    except:
#        StepAError.append(Page)
#        continue
        
    #Step B
    try:
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')
        from Split import VertPart

        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import Filter, ConvertDict
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
        from PageCut import HoriPart

        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)

        HoriPoint=HoriPart(img,Page,texts)[0]

        try:
            VertPoint=VertPart(path)[1]
            print('Failed detecting Vertical Point')
            HH,WW=img.shape[:2]
            VertPoint=1*HH//3
        except:
            print('Failed detecting Vertical Point')
            HH,WW=img.shape[:2]
            VertPoint=HH//2

        ##Right Page
        BoxR=Filter(BookPage,texts,HoriPoint)
        BoxR=ConvertDict(BoxR)

        ##Left Page
        BoxL=Filter(BookPage+1,texts,HoriPoint)
        BoxL=ConvertDict(BoxL)
        
        Dict={}
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from Organize import FilterBox
        sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
        from DetectOffice import DetectOffice
        LocList=[]

        #RightPage
        OfficeList=Rows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxR, Office,VertPoint))
            BoxR=FilterBox(BoxR,LocList,VertPoint)

        #LeftPage
        OfficeList=NxRows['Office'].tolist()
        for Office in OfficeList:
            LocList.append(DetectOffice(BoxL, Office,VertPoint))
            BoxL=FilterBox(BoxL,LocList,VertPoint)

        Dict[Page]=LocList
    except:
        StepBError.append(Page)
        continue
        
    # Main Code 
#    try:        
    #Split quardrant into offices and detect Positions
    sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Position')
    from OrganizePosi import Split, SplitClova, Crop, CropClova
    from DetectPosi import DetectPosi,DetectPosiClova,AggregatePosi,RefPosiDict,RefPosiDict2
    from Layout import RefineVert
    
    PosiDict,PosiDict_CLOVA={},{}
    CrossWalk=pd.read_csv(origin+"Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
    Positions=CrossWalk.tolist()
    PosiDict['Pre']=[]
    PosiDict_CLOVA['Pre']=[]

    DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)
    DF_CLOVA=CropClova(textsCLOVA,HoriPoint,VertPoint,Dict,Page)

    ##UR
    BoxList,BoxListCLOVA=Split(DF['UR']['Box'],DF['UR']['Thres']),SplitClova(DF_CLOVA['UR']['Box'],DF_CLOVA['UR']['Thres'])
    PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UR']['Thres'],PosiDict_CLOVA,Positions)

    ##LR
    BoxList,BoxListCLOVA=Split(DF['LR']['Box'],DF['LR']['Thres']),SplitClova(DF_CLOVA['LR']['Box'],DF_CLOVA['LR']['Thres'])
    PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LR']['Thres'],PosiDict_CLOVA,Positions)

    ##UL
    BoxList,BoxListCLOVA=Split(DF['UL']['Box'][1:],DF['UL']['Thres']),SplitClova(DF_CLOVA['UL']['Box'],DF_CLOVA['UL']['Thres'])
    PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UL']['Thres'],PosiDict_CLOVA,Positions)

    #LL
    BoxList,BoxListCLOVA=Split(DF['LL']['Box'],DF['LL']['Thres']),SplitClova(DF_CLOVA['LL']['Box'],DF_CLOVA['LL']['Thres'])
    PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LL']['Thres'],PosiDict_CLOVA,Positions)

    FinalPosiDict=AggregatePosi(PosiDict,PosiDict_CLOVA)

    try:
        FinalPosiDict=RefPosiDict(texts,FinalPosiDict)
    except:
        MainError.append(list([Page,]))
        continue

    try:
        VertPoint2=RefineVert(img,LocList,FinalPosiDict,VertPoint,HoriPoint)
    except:
        MainError.append(Page)
        continue

    try:
        FinalPosiDict=RefPosiDict2(FinalPosiDict,VertPoint,VertPoint2)
    except:
        MainError.append(Page)
        continue

    Data.append(FinalPosiDict)

    #
#    except:
#        MainError.append(Page)

Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
Failed detecting Vertical Point
Failed detecting Vertical Point
Failed detecting Vertical Point
Horizontal Line was not automatically detected... Defining line arbitrariry...
Failed detecting Vertical Point
No Office at Left Side. Page117
Failed detecting Vertical Point
Failed detecting Vertical Point
No Office at Left Side. Page157
Failed detecting Vertical Point
No Office at Right Side. Page176
Failed detecting Vertical Point
No Office at Right Side. Page196
Failed detecting Vertical Point
No Office at Left Side. Page217
Failed detecting Vertical Point


# Summary of performance

**1. Non-Running Error**

In [98]:
from ShowPosi import Show

def ErrorRate(ErrorList,PageList):
    return(round(len(ErrorList)/len(PageList),2))

In [99]:
PageList=range(10, 120, 5)
ErrorRate(StepAError,PageList),ErrorRate(StepBError,PageList),ErrorRate(MainError,PageList)

(0.0, 0.0, 0.14)

In [100]:
StepBError

[]

In [101]:
MainError

[[35], [55], [65]]

**2.Miss Assignment Error**

In [105]:
for PosiDict in Data:
    for Office in list(PosiDict.keys()):
        if  Office == 'Pre':
            continue
        BookPage=df[df['Office']==Office]['Page'].tolist()[0]
        if BookPage%2==0:
            Page=(BookPage+14)//2
        else:
            Page=(BookPage+13)//2
    
        file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.jpg'
        path=os.path.join(origin,file_path)
        img=cv2.imread(path)
        
        HoriPoint=HoriPart(img,Page,texts)[0]
        VertPoint2=1000
        
        Show(PosiDict,Office,img,VertPoint2,HoriPoint)